# Qwen3-0.6B 明日方舟干员助手 LoRA 微调

基于 Qwen3-0.6B 模型，使用明日方舟干员数据集（449个干员，22621条问答）进行 LoRA 微调，打造一个能回答干员问题、扮演干员对话的专属助手。

**数据集**：[arknights-dataset](https://github.com/RainmeoX/arknights-dataset)

**环境**：AMD Radeon Cloud（ROCm 单卡）

**训练监控**：TensorBoard（无需登录）

## 1. 环境准备

In [ ]:
# 安装依赖（国内源）
%pip install -q transformers datasets peft accelerate modelscope tensorboard -i https://mirrors.cloud.tencent.com/pypi/simple/

In [ ]:
import torch
from datasets import Dataset
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer

# 检查 GPU
print('PyTorch:', torch.__version__)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('显存:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GB')

## 2. 下载模型和数据集

In [ ]:
# 从 ModelScope 下载 Qwen3-0.6B
from modelscope import snapshot_download

model_path = snapshot_download('Qwen/Qwen3-0.6B', cache_dir='./models')
print(f'模型路径: {model_path}')

In [ ]:
# 克隆数据集仓库（如果还没有）
import os
if not os.path.exists('arknights-dataset'):
    !git clone https://github.com/RainmeoX/arknights-dataset.git
else:
    print('数据集已存在')

In [ ]:
# 加载数据集
train_data = []
with open('arknights-dataset/data/train.jsonl', encoding='utf-8') as f:
    for line in f:
        train_data.append(eval(line))

print(f'训练集大小: {len(train_data)}')
print('示例:')
for item in train_data[:3]:
    print(f"  Q: {item['instruction']}")
    print(f"  A: {item['output'][:80]}")
    print()

## 3. 数据预处理

In [ ]:
# 转换为 Dataset 对象
df = pd.DataFrame(train_data)
ds = Dataset.from_pandas(df)
ds

In [ ]:
# 加载 tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
tokenizer

In [ ]:
# 查看 chat_template 格式
messages = [
    {"role": "system", "content": "你是明日方舟游戏助手"},
    {"role": "user", "content": "银灰是什么职业？"},
    {"role": "assistant", "content": "银灰是6星近卫干员。"},
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
print(text)

In [ ]:
# 数据处理函数
SYSTEM_PROMPT = "你是明日方舟游戏助手，可以回答关于干员的各种问题，也能扮演干员进行对话。"

def process_func(example):
    MAX_LENGTH = 1024
    input_ids, attention_mask, labels = [], [], []
    
    # 构造 chat 格式
    instruction = tokenizer(
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{example['instruction']}<|im_end|>\n"
        f"<|im_start|>assistant\n",
        add_special_tokens=False
    )
    response = tokenizer(
        f"{example['output']}<|im_end|>\n",
        add_special_tokens=False
    )
    
    input_ids = instruction["input_ids"] + response["input_ids"] + [tokenizer.eos_token_id]
    attention_mask = instruction["attention_mask"] + response["attention_mask"] + [1]
    labels = [-100] * len(instruction["input_ids"]) + response["input_ids"] + [tokenizer.eos_token_id]
    
    # 截断
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [ ]:
# 处理数据集
tokenized_id = ds.map(process_func, remove_columns=ds.column_names)
tokenized_id

In [ ]:
# 验证处理结果
print('处理后的第一条数据:')
print('input_ids 长度:', len(tokenized_id[0]['input_ids']))
print('解码:', tokenizer.decode(tokenized_id[0]['input_ids']))

## 4. 加载模型

In [ ]:
import torch

model = AutoModelForCausalLM.from_pretrained(
    model_path, 
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="eager"
)
model.config.use_cache = True
model

## 5. 配置 LoRA

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    inference_mode=False,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1
)
config

In [ ]:
model = get_peft_model(model, config)
model.print_trainable_parameters()

## 6. 训练配置（最大化性能）

In [ ]:
args = TrainingArguments(
    output_dir="./output/Qwen3_Arknights_LoRA",
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    logging_steps=10,
    num_train_epochs=5,
    save_steps=500,
    learning_rate=2e-4,
    save_on_each_node=True,
    gradient_checkpointing=False,
    bf16=True,
    dataloader_num_workers=4,
    report_to="none",
)

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_id,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
)

## 7. 开始训练

训练约 11 分钟，可在终端运行 `tensorboard --logdir ./logs` 查看训练曲线。

In [ ]:
trainer.train()

## 8. 测试模型

In [ ]:
# 测试对话
def chat(question, system_prompt=SYSTEM_PROMPT):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True,
        enable_thinking=False,
    ).to(model.device)
    
    gen_kwargs = {"max_new_tokens": 512, "do_sample": False, "top_k": 1}
    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_kwargs)
        outputs = outputs[:, inputs['input_ids'].shape[1]:]
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# 测试几个问题
questions = [
    "银灰是什么职业的干员？",
    "介绍一下阿米娅这名干员",
    "扮演银灰，说一句任命助理的台词",
    "6星医疗干员有哪些？",
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"问：{q}")
    print(f"答：{chat(q)}")

## 9. 保存模型

In [ ]:
# 保存 LoRA adapter
trainer.save_model("./output/Qwen3_Arknights_LoRA_final")
tokenizer.save_pretrained("./output/Qwen3_Arknights_LoRA_final")
print('模型已保存到 ./output/Qwen3_Arknights_LoRA_final')

## 10. 重新加载模型推理

In [ ]:
import torch

model = AutoModelForCausalLM.from_pretrained(
    model_path, 
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="eager"
)
model.config.use_cache = True
model

In [ ]:
# 最终测试
questions = [
    "介绍一下银灰这名干员",
    "扮演银灰，说一句任命助理的台词",
    "6星医疗干员有哪些？",
    "星熊的满级属性是多少？",
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"问：{q}")
    print(f"答：{chat(q)}")

## 11. 运行评估（可选）

训练完成后，可以运行评估脚本测试模型效果：

```bash
# 1. 生成预测结果
python generate_predictions.py \
    --base_model_path {model_path} \
    --lora_path ./output/Qwen3_Arknights_LoRA_final \
    --test_dir arknights-dataset/eval/ \
    --output predictions.json

# 2. 运行评估
python arknights-dataset/eval/evaluate.py \
    --predictions predictions.json \
    --test_dir arknights-dataset/eval/ \
    --output eval_report.json \
    --fluency_score 85
```